In [2]:
import os
from datetime import datetime
import pandas as pd

In [5]:
df_orders = pd.read_csv('df_orders.csv')
df_orders.head()

,Unnamed: 0,courier_id,group,time
0,1038,0,pilot,887
1,1072,0,pilot,698
2,1071,0,pilot,960
3,1070,0,pilot,776
4,1069,0,pilot,533


In [8]:
mean_time_pilot = df_orders[df_orders["group"] == 'pilot'].time.mean()
print(mean_time_pilot)

704.5489239598278


In [9]:
mean_time_control = df_orders[df_orders["group"] == 'control'].time.mean()

In [10]:
print(mean_time_pilot)

704.5489239598278


In [11]:
diff = mean_time_pilot - mean_time_control

In [12]:
print(diff)

-40.16858037292786


In [17]:
import pandas as pd
from scipy.stats import mannwhitneyu

df_orders = pd.read_csv('df_orders.csv')

# Коэффициент — среднее время заказа в контрольной группе
kappa = df_orders.loc[
    df_orders['group'] == 'control', 'time'
].mean()

# Агрегация до уровня единицы рандомизации — курьера
df_linearized = (
    df_orders
    .groupby(['courier_id', 'group'])
    .agg(
        sum_time=('time', 'sum'),
        count_orders=('time', 'count')
    )
    .reset_index()
)

df_linearized['linearized_time'] = (
    df_linearized['sum_time']
    - kappa * df_linearized['count_orders']
)

pilot = df_linearized.loc[
    df_linearized['group'] == 'pilot',
    'linearized_time'
]

control = df_linearized.loc[
    df_linearized['group'] == 'control',
    'linearized_time'
]

result = mannwhitneyu(pilot, control, alternative='two-sided')
print(result.pvalue)

1.104471211676822e-05
